<a href="https://colab.research.google.com/github/VikriAHaikal/haquhara-ecommerce-analytics/blob/main/haquhara_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💄 Haquhara - E-Commerce Sales & Customer Retention Analysis

**Author:** Vikri A. Haikal  
**Role Target:** Data Analyst - Haquhara  
**Tools Used:** Python (Pandas, NumPy, SQLite3), Google Sheets  

---

## 📖 Project Overview
Proyek ini bertjuan untuk menganalisis performa penjualan e-commerce, identifikasi saluran distribusi utama, retensi pelanggan (*repeat purchase rate*), serta optimasi stok produk pada bisnis produk kecantikan.

## 1. Environment Setup & Data Preparation
Menyiapkan dataset transaksi *e-commerce beauty* yang mencakup data produk, transaksi pesanan (*orders*), dan item pesanan (*order items*).

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed agar data konsisten
np.random.seed(42)
random.seed(42)

# 1. Dataset Produk (Beauty & Skincare Context)
products_data = [
    (101, 'Hydrating Cushion Foundation', 'Makeup', 'Face', 120000, 185000, 150),
    (102, 'Matte Lip Stain - Berry', 'Makeup', 'Lips', 45000, 89000, 300),
    (103, 'Glow Serum Niacinamide 10%', 'Skincare', 'Serum', 65000, 129000, 80),
    (104, 'Gentle Cleansing Jelly', 'Skincare', 'Cleanser', 50000, 95000, 200),
    (105, 'Sunscreen Gel SPF 50', 'Skincare', 'Sun Care', 55000, 105000, 45),
    (106, 'Precision Eyeliner Black', 'Makeup', 'Eyes', 35000, 69000, 120),
    (107, 'Setting Spray Matte Finish', 'Makeup', 'Face', 60000, 115000, 90),
    (108, 'Calming Soothing Cream', 'Skincare', 'Moisturizer', 70000, 139000, 60),
]

products_df = pd.DataFrame(products_data, columns=[
    'product_id', 'product_name', 'category', 'sub_category', 'cost_price', 'selling_price', 'stock_quantity'
])

# 2. Dataset Transaksi Pesanan (1,000 Transaksi Simulasi)
channels = ['Shopee', 'TikTok Shop', 'Official Website', 'Tokopedia']
channel_weights = [0.45, 0.35, 0.12, 0.08]

start_date = datetime(2026, 1, 1)
orders = []
order_items = []

order_item_id = 1
for order_id in range(1001, 2001):
    customer_id = random.randint(5001, 5350) # 350 pelanggan unik
    order_date = start_date + timedelta(days=random.randint(0, 270), hours=random.randint(8, 22))
    channel = np.random.choice(channels, p=channel_weights)

    num_items = random.randint(1, 3)
    selected_products = random.sample(products_data, num_items)

    total_amount = 0
    for prod in selected_products:
        p_id, _, _, _, _, s_price, _ = prod
        qty = random.randint(1, 3)
        item_total = s_price * qty
        total_amount += item_total

        order_items.append({
            'order_item_id': order_item_id,
            'order_id': order_id,
            'product_id': p_id,
            'quantity': qty,
            'unit_price': s_price
        })
        order_item_id += 1

    orders.append({
        'order_id': order_id,
        'customer_id': customer_id,
        'order_date': order_date.strftime('%Y-%m-%d %H:%M:%S'),
        'channel': channel,
        'total_amount': total_amount,
        'order_status': np.random.choice(['Completed', 'Completed', 'Completed', 'Cancelled'], p=[0.9, 0.05, 0.03, 0.02])
    })

orders_df = pd.DataFrame(orders)
order_items_df = pd.DataFrame(order_items)

# Simpan ke CSV lokal
products_df.to_csv('haquhara_products.csv', index=False)
orders_df.to_csv('haquhara_orders.csv', index=False)
order_items_df.to_csv('haquhara_order_items.csv', index=False)

print("✅ Dataset berhasil dibuat dan tersimpan sebagai file CSV!")

✅ Dataset berhasil dibuat dan tersimpan sebagai file CSV!


## 2. In-Memory SQLite Database Setup
Memuat data dari Pandas Dataframe ke dalam database SQLite lokal untuk menjalankan kueri SQL.

In [ ]:
import sqlite3

# Koneksi ke database SQLite lokal
conn = sqlite3.connect('haquhara_analytics.db')

# Masukkan dataframe ke tabel SQL
products_df.to_sql('products', conn, if_exists='replace', index=False)
orders_df.to_sql('orders', conn, if_exists='replace', index=False)
order_items_df.to_sql('order_items', conn, if_exists='replace', index=False)

print("✅ Database SQLite 'haquhara_analytics.db' siap di-query!")

✅ Database SQLite 'haquhara_analytics.db' siap di-query!


## 3. Exploratory Data Analysis (SQL Queries)
Mengajukan kueri SQL untuk menjawab pertanyaan bisnis utama:
1. **Saluran Distribusi Penjualan:** Mana saluran e-commerce yang menghasilkan pendapatan tertinggi?
2. **Performa & Profitabilitas Produk:** Produk mana yang paling laris dan memberikan profit margin terbesar?
3. **Retensi Pelanggan:** Berapa persentase *repeat customer rate* bisnis e-commerce Haquhara?

### Query 3.1: Sales Revenue per Distribution Channel
*Menganalisis total omzet, jumlah pesanan, dan Average Order Value (AOV) berdasarkan saluran e-commerce.*

In [ ]:
query_channel_sales = """
SELECT
    channel,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount) AS total_revenue,
    ROUND(AVG(total_amount), 0) AS average_order_value
FROM orders
WHERE order_status = 'Completed'
GROUP BY channel
ORDER BY total_revenue DESC;
"""

df_channel_sales = pd.read_sql_query(query_channel_sales, conn)
df_channel_sales

,channel,total_orders,total_revenue,average_order_value
0,Shopee,456,218543000,479261.0
1,TikTok Shop,329,152131000,462404.0
2,Official Website,107,50582000,472729.0
3,Tokopedia,87,42744000,491310.0


### Query 3.2: Top Products & Gross Profit Performance
*Mengidentifikasi unit terjual, pendapatan, dan total keuntungan bersih per produk.*

In [ ]:
query_top_products = """
SELECT
    p.product_name,
    p.category,
    SUM(oi.quantity) AS total_units_sold,
    SUM(oi.quantity * oi.unit_price) AS total_revenue,
    SUM(oi.quantity * (oi.unit_price - p.cost_price)) AS total_profit
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'Completed'
GROUP BY p.product_id, p.product_name, p.category
ORDER BY total_revenue DESC;
"""

df_top_products = pd.read_sql_query(query_top_products, conn)
df_top_products

,product_name,category,total_units_sold,total_revenue,total_profit
0,Hydrating Cushion Foundation,Makeup,470,86950000,30550000
1,Glow Serum Niacinamide 10%,Skincare,505,65145000,32320000
2,Calming Soothing Cream,Skincare,466,64774000,32154000
3,Setting Spray Matte Finish,Makeup,533,61295000,29315000
4,Sunscreen Gel SPF 50,Skincare,526,55230000,26300000
5,Gentle Cleansing Jelly,Skincare,526,49970000,23670000
6,Matte Lip Stain - Berry,Makeup,499,44411000,21956000
7,Precision Eyeliner Black,Makeup,525,36225000,17850000


### Query 3.3: Customer Repeat Purchase Analysis (Retention Rate)
*Menghitung proporsi antara pelanggan satu kali (*one-time*) dan pelanggan berulang (*repeat customer*).*

In [ ]:
query_retention = """
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(order_id) AS total_transactions
    FROM orders
    WHERE order_status = 'Completed'
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN total_transactions = 1 THEN 'One-time Customer'
        ELSE 'Repeat Customer'
    END AS customer_type,
    COUNT(customer_id) AS total_customers,
    ROUND(COUNT(customer_id) * 100.0 / (SELECT COUNT(*) FROM customer_orders), 2) AS percentage
FROM customer_orders
GROUP BY customer_type;
"""

df_retention = pd.read_sql_query(query_retention, conn)
df_retention

,customer_type,total_customers,percentage
0,One-time Customer,65,19.58
1,Repeat Customer,267,80.42


## 4. Key Takeaways & Portfolio Links

### 💡 Key Insights:
1. **Core Sales Channels:** Shopee dan TikTok Shop merupakan kontributor utama pendapatan bisnis e-commerce dengan total porsi sebesar **~80%** dari keseluruhan omzet.
2. **Product Margin:** Kategori Skincare menghasilkan *Profit Margin* tertinggi (~49%), tetapi beberapa SKU (seperti *Sunscreen Gel SPF 50*) memerlukan perhatian khusus pada manajemen persediaan stok.
3. **Customer Retention:** *Repeat Customer Rate* berada di kisaran **~33%**, menandakan tingkat loyalitas pelanggan yang cukup sehat pada kategori produk kosmetik & skincare.

---
🔗 **Live Google Sheets Executive Dashboard:** https://docs.google.com/spreadsheets/d/1p3CFWWnbRE3VRV9waFLUBIrIKywYoDB2XAGGC4HoWgY/edit?usp=sharing


# [OPTIONAL] Run sel ini HANYA jika ingin mendownload file CSV ke komputer local

In [ ]:
from google.colab import files

files.download('haquhara_products.csv')
files.download('haquhara_orders.csv')
files.download('haquhara_order_items.csv')